<a href="https://colab.research.google.com/github/mohsinimam/AIinHealth/blob/main/Mohsin_HealthcareAI_LLM_HW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# AI IN HEALTHCARE: LLM TUTORIAL COLAB NOTEBOOK
# Topic: Predicting Diabetes Risk using Generative AI (Gemini)
# Requirements met: Zero-shot, Few-shot, Chain-of-Thought, Tree of Thoughts,
#                   and Embeddings + Logistic Regression (Bonus Point).


Install Key dependencies

In [1]:
!pip install -q google-generativeai scikit-learn pandas tqdm matplotlib

Import Settings

In [7]:
import os
import time
import pandas as pd
import numpy as np
import google.generativeai as genai
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report

API Configuration

In [8]:
from google.colab import userdata
try:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except:
    # Fallback for local testing
    GEMINI_API_KEY = "YOUR_GEMINI_API_KEY_HERE"

genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel('gemini-2.5-flash')

Load Patients and Observation data from the synthea Dataset

In [13]:
# create a dataframe and read in csv file
patients_df = pd.read_csv('/patients.csv')
observations_df = pd.read_csv('/observations.csv')

In [16]:
#show top rows from the dataframe just created
patients_df.head()

,Id,BIRTHDATE,DEATHDATE,SSN,DRIVERS,PASSPORT,PREFIX,FIRST,MIDDLE,LAST,...,CITY,STATE,COUNTY,FIPS,ZIP,LAT,LON,HEALTHCARE_EXPENSES,HEALTHCARE_COVERAGE,INCOME
0,f1aa52b9-aded-3188-9386-012244805ebf,1989-06-20,NaN,999-96-7640,S99993266,X52137231X,Mr.,Maurice742,Corey514,Brekke496,...,Dedham,Massachusetts,Norfolk County,25021.0,2090,42.294422,-71.141595,182949.51,22601.32,108063
1,d30ace70-ad74-f9a6-2433-a5f28a25a03d,1970-06-07,NaN,999-80-5045,S99934663,X57379994X,Mrs.,Débora815,Verónica383,Montes106,...,Woburn,Massachusetts,Middlesex County,25017.0,1890,42.497181,-71.193682,350449.26,738677.48,30032
2,cd42752c-2467-db64-102c-73d9b3b4f218,1990-03-23,NaN,999-58-3815,S99960338,X2841412X,Mrs.,Lekisha909,Lurline371,Bosco882,...,Natick,Massachusetts,Middlesex County,NaN,0,42.273299,-71.303382,181444.45,1175449.91,132744
3,182b5fa5-9a66-b61f-0ca4-0106c17f92a8,1954-03-26,NaN,999-37-7227,S99917941,X36873868X,Mrs.,Berta524,Carmen818,Escobar593,...,Saugus,Massachusetts,Essex County,25009.0,1906,42.515774,-71.017220,778776.76,117169.66,78140
4,e972e250-fba2-a39f-81e9-25f7df4f83a2,2017-05-12,NaN,999-22-5525,NaN,NaN,NaN,Bryon392,Reid278,Runte676,...,Chelmsford,Massachusetts,Middlesex County,NaN,0,42.620652,-71.365034,1616.96,26415.77,12994


In [15]:
observations_df.head()

,DATE,PATIENT,ENCOUNTER,CATEGORY,CODE,DESCRIPTION,VALUE,UNITS,TYPE
0,2016-05-10T13:10:24Z,f1aa52b9-aded-3188-9386-012244805ebf,f1aa52b9-aded-3188-0937-f004b659f1b5,vital-signs,8302-2,Body Height,185.2,cm,numeric
1,2016-05-10T13:10:24Z,f1aa52b9-aded-3188-9386-012244805ebf,f1aa52b9-aded-3188-0937-f004b659f1b5,vital-signs,72514-3,Pain severity - 0-10 verbal numeric rating [Sc...,2.0,{score},numeric
2,2016-05-10T13:10:24Z,f1aa52b9-aded-3188-9386-012244805ebf,f1aa52b9-aded-3188-0937-f004b659f1b5,vital-signs,29463-7,Body Weight,78.8,kg,numeric
3,2016-05-10T13:10:24Z,f1aa52b9-aded-3188-9386-012244805ebf,f1aa52b9-aded-3188-0937-f004b659f1b5,vital-signs,39156-5,Body mass index (BMI) [Ratio],23.0,kg/m2,numeric
4,2016-05-10T13:10:24Z,f1aa52b9-aded-3188-9386-012244805ebf,f1aa52b9-aded-3188-0937-f004b659f1b5,vital-signs,8462-4,Diastolic Blood Pressure,75.0,mm[Hg],numeric


## Data Processing

1. Extract PatientID and Calculate Age from patients_df
Synthea uses 'Id' for the patient identifier and 'BIRTHDATE' for birth date

In [29]:
patients_df['BIRTHDATE'] = pd.to_datetime(patients_df['BIRTHDATE'])
patients_df['Age'] = (pd.to_datetime('today') - patients_df['BIRTHDATE']).dt.days // 365
df_base = patients_df[['Id', 'Age']].rename(columns={'Id': 'PatientID'})

2. Extract BMI, Systolic BP, and A1C from observations_df

In [30]:
observations_df['DATE'] = pd.to_datetime(observations_df['DATE'])
target_descriptions = [
    'Body mass index (BMI) [Ratio]',
    'Systolic Blood Pressure',
    'Hemoglobin A1c/Hemoglobin.total in Blood'
]

Filter for the specific biomarkers, convert values, and get the most recent reading per patient

In [31]:
obs_filtered = observations_df[observations_df['DESCRIPTION'].isin(target_descriptions)].copy()
obs_filtered['VALUE'] = pd.to_numeric(obs_filtered['VALUE'], errors='coerce')
obs_filtered = obs_filtered.sort_values('DATE').drop_duplicates(subset=['PATIENT', 'DESCRIPTION'], keep='last')

Pivot so each metric becomes a column

In [32]:
obs_pivot = obs_filtered.pivot(index='PATIENT', columns='DESCRIPTION', values='VALUE').reset_index()
obs_pivot = obs_pivot.rename(columns={
    'PATIENT': 'PatientID',
    'Body mass index (BMI) [Ratio]': 'BMI',
    'Systolic Blood Pressure': 'Systolic_BP',
    'Hemoglobin A1c/Hemoglobin.total in Blood': 'A1C_Level'
})

3. Merge Patients and Observations

In [33]:
df = pd.merge(df_base, obs_pivot, on='PatientID', how='inner')

Drop any patients missing these specific lab values and format the numbers

In [34]:
df = df.dropna().copy()
df['BMI'] = np.round(df['BMI'], 1)
df['Systolic_BP'] = np.round(df['Systolic_BP'], 0).astype(int)
df['A1C_Level'] = np.round(df['A1C_Level'], 1)

Create Ground Truth: A1C >= 6.0 is typically indicative of Diabetes

In [42]:
df['Diabetes_Risk'] = (df['A1C_Level'] >= 6.0).astype(int)

Test

In [43]:
df.head()

,PatientID,Age,BMI,A1C_Level,Systolic_BP,Diabetes_Risk
1,d30ace70-ad74-f9a6-2433-a5f28a25a03d,55,29.0,6.2,131,1
2,cd42752c-2467-db64-102c-73d9b3b4f218,36,30.2,6.1,142,1
3,182b5fa5-9a66-b61f-0ca4-0106c17f92a8,72,27.5,5.9,90,0
9,0c193146-985b-c33e-07fe-6f474df3336c,59,28.6,6.3,146,1
11,d7c1fce2-02be-45de-b6a5-d4a7222ad1dd,45,29.8,5.9,129,0


In [44]:
#show df entries where Diabetes_Risk is 1
df[df['Diabetes_Risk'] == 1].head()

,PatientID,Age,BMI,A1C_Level,Systolic_BP,Diabetes_Risk
1,d30ace70-ad74-f9a6-2433-a5f28a25a03d,55,29.0,6.2,131,1
2,cd42752c-2467-db64-102c-73d9b3b4f218,36,30.2,6.1,142,1
9,0c193146-985b-c33e-07fe-6f474df3336c,59,28.6,6.3,146,1
12,a4f0c168-02ff-13b6-2c68-6a21d440915e,57,30.2,6.4,116,1
13,08c6ca9c-1ba6-97b5-0c33-65f7755af0e2,62,30.4,6.1,137,1


In [45]:
#total entries in dataframe
len(df)

54

In [46]:
#show lenght of entries with diabetes risk 1 and those with diabetes risk 2
print(len(df[df['Diabetes_Risk'] == 1]))

38


# helper function

In [47]:
def create_patient_text(row):
    return (f"Patient {row['PatientID']} is {row['Age']} years old, has a BMI of {row['BMI']}, "
            f"a Systolic Blood Pressure of {row['Systolic_BP']}, and an A1C level of {row['A1C_Level']}%.")

create column patient test

In [48]:
df['Patient_Text'] = df.apply(create_patient_text, axis=1)

In [49]:
df.head()

,PatientID,Age,BMI,A1C_Level,Systolic_BP,Diabetes_Risk,Patient_Text
1,d30ace70-ad74-f9a6-2433-a5f28a25a03d,55,29.0,6.2,131,1,Patient d30ace70-ad74-f9a6-2433-a5f28a25a03d i...
2,cd42752c-2467-db64-102c-73d9b3b4f218,36,30.2,6.1,142,1,Patient cd42752c-2467-db64-102c-73d9b3b4f218 i...
3,182b5fa5-9a66-b61f-0ca4-0106c17f92a8,72,27.5,5.9,90,0,Patient 182b5fa5-9a66-b61f-0ca4-0106c17f92a8 i...
9,0c193146-985b-c33e-07fe-6f474df3336c,59,28.6,6.3,146,1,Patient 0c193146-985b-c33e-07fe-6f474df3336c i...
11,d7c1fce2-02be-45de-b6a5-d4a7222ad1dd,45,29.8,5.9,129,0,Patient d7c1fce2-02be-45de-b6a5-d4a7222ad1dd i...


# Split Train and Test Data - 80 - 20

In [50]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

print(f"Generated {len(df)} synthetic patient records.")
print("Sample Patient Text:", df['Patient_Text'].iloc[0])

Generated 54 synthetic patient records.
Sample Patient Text: Patient d30ace70-ad74-f9a6-2433-a5f28a25a03d is 55 years old, has a BMI of 29.0, a Systolic Blood Pressure of 131, and an A1C level of 6.2%.


# IN-CONTEXT LEARNING & REASONING (PROMPT ENGINEERING)

In [53]:
# Let's take a single patient from the test set to demonstrate the different prompting methods
sample_patient = test_df.iloc[6]['Patient_Text']
actual_status = "Positive" if test_df.iloc[6]['Diabetes_Risk'] == 1 else "Negative"

print(f"\n--- Testing on Patient: {actual_status} Ground Truth ---")

# --- METHOD A: Zero-Shot Prompting ---
zero_shot_prompt = f"""
You are an expert endocrinologist.
Determine if the following patient is at high risk for diabetes.
Reply with only a single word: "Positive" or "Negative".

{sample_patient}
Synergy/Risk:
"""
response_zero = model.generate_content(zero_shot_prompt).text.strip()
print(f"Zero-Shot Result: {response_zero}")


# --- METHOD B: Few-Shot Prompting ---
few_shot_prompt = f"""
You are an expert endocrinologist. Determine if the patient is at high risk for diabetes.

Patient: Patient A is 45 years old, has a BMI of 24.5, a Systolic Blood Pressure of 120, and an A1C level of 5.2%.
Risk: Negative

Patient: Patient B is 62 years old, has a BMI of 32.1, a Systolic Blood Pressure of 145, and an A1C level of 7.1%.
Risk: Positive

{sample_patient}
Risk:
"""
response_few = model.generate_content(few_shot_prompt).text.strip()
print(f"Few-Shot Result: {response_few}")


# --- METHOD C: Chain-of-Thought (CoT) Prompting ---
cot_prompt = f"""
You are an expert endocrinologist. Evaluate the following patient for diabetes risk.
{sample_patient}

Let's think step by step.
1. Evaluate the A1C level (normal is below 5.7%, prediabetes is 5.7-6.4%, diabetes is 6.5% or higher).
2. Consider compounding factors like BMI and Age.
3. Conclude with a final determination of "Positive" or "Negative".
"""
response_cot = model.generate_content(cot_prompt).text.strip()
print(f"\nChain-of-Thought Result:\n{response_cot}")

# --- METHOD D: Tree of Thoughts (ToT) Prompting ---
tot_prompt = f"""
Imagine three different medical experts are answering this question: Is this patient at high risk for diabetes?
{sample_patient}

All experts will write down 1 step of their thinking, then share it with the group.
Then all experts will go on to the next step. If any expert realizes they are wrong, they leave.
Finally, the remaining experts agree on a single word conclusion: "Positive" or "Negative".
"""
response_tot = model.generate_content(tot_prompt).text.strip()
print(f"\nTree of Thoughts Result:\n{response_tot}")


--- Testing on Patient: Positive Ground Truth ---
Zero-Shot Result: Positive
Few-Shot Result: Risk: Positive

Chain-of-Thought Result:
Let's evaluate the patient's diabetes risk step by step:

1.  **Evaluate the A1C level:**
    *   The patient's A1C is 6.1%.
    *   According to the American Diabetes Association (ADA) guidelines:
        *   Below 5.7% is considered normal.
        *   5.7% to 6.4% is considered prediabetes.
        *   6.5% or higher is considered diabetes.
    *   Therefore, an A1C of 6.1% places the patient in the **prediabetes** category. This signifies a significantly increased risk for developing type 2 diabetes.

2.  **Consider compounding factors:**
    *   **BMI:** The patient's BMI is 30.1. A BMI of 30 or higher is classified as **obesity (Class I)**. Obesity is a major, modifiable risk factor for type 2 diabetes.
    *   **Age:** The patient is 73 years old. Increasing age is an independent, non-modifiable risk factor for type 2 diabetes.
    *   **Systoli

In [55]:
# To properly evaluate the effectiveness of the prompting methods, we create a test harness.
eval_df = test_df.head(15).copy()

print(f"\n--- Running Prompt Engineering Evaluation on {len(eval_df)} Patients ---")
print("This may take a minute due to API rate limits...\n")

# Helper function to robustly parse the LLM's text response into a binary 1 or 0
def parse_llm_output(text):
    text_lower = text.lower()
    # Check for the specific exact-match strings we requested in CoT and ToT
    if "final conclusion: positive" in text_lower: return 1
    if "final conclusion: negative" in text_lower: return 0

    # Fallback: check the last few words generated by the model
    last_words = " ".join(text_lower.split()[-10:])
    if "positive" in last_words: return 1
    if "negative" in last_words: return 0

    # Ultimate fallback: check if the word exists anywhere in the text
    return 1 if "positive" in text_lower else 0

# Dictionary to store predictions for metric calculation
prompt_results = {
    'Actual': eval_df['Diabetes_Risk'].tolist(),
    'Zero-Shot': [],
    'Few-Shot': [],
    'CoT': [],
    'ToT': []
}

for idx, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Evaluating Prompts"):
    patient_text = row['Patient_Text']

    # --- METHOD A: Zero-Shot Prompting ---
    zero_shot_prompt = f"""
    You are an expert endocrinologist.
    Determine if the following patient is at high risk for diabetes.
    Reply with only a single word: "Positive" or "Negative".

    {patient_text}
    Risk:
    """
    response_zero = model.generate_content(zero_shot_prompt).text.strip()
    prompt_results['Zero-Shot'].append(parse_llm_output(response_zero))
    time.sleep(0.5) # Buffer to prevent hitting free-tier API rate limits

    # --- METHOD B: Few-Shot Prompting ---
    few_shot_prompt = f"""
    You are an expert endocrinologist. Determine if the patient is at high risk for diabetes.

    Patient: Patient A is 45 years old, has a BMI of 24.5, a Systolic Blood Pressure of 120, and an A1C level of 5.2%.
    Risk: Negative

    Patient: Patient B is 62 years old, has a BMI of 32.1, a Systolic Blood Pressure of 145, and an A1C level of 7.1%.
    Risk: Positive

    {patient_text}
    Risk:
    """
    response_few = model.generate_content(few_shot_prompt).text.strip()
    prompt_results['Few-Shot'].append(parse_llm_output(response_few))
    time.sleep(0.5)

    # --- METHOD C: Chain-of-Thought (CoT) Prompting ---
    cot_prompt = f"""
    You are an expert endocrinologist. Evaluate the following patient for diabetes risk.
    {patient_text}

    Let's think step by step.
    1. Evaluate the A1C level (normal is below 5.7%, prediabetes is 5.7-6.4%, diabetes is 6.5% or higher).
    2. Consider compounding factors like BMI and Age.
    3. Conclude with a final determination on a new line exactly like this: "Final Conclusion: Positive" or "Final Conclusion: Negative".
    """
    response_cot = model.generate_content(cot_prompt).text.strip()
    prompt_results['CoT'].append(parse_llm_output(response_cot))
    time.sleep(0.5)

    # --- METHOD D: Tree of Thoughts (ToT) Prompting ---
    tot_prompt = f"""
    Imagine three different medical experts are answering this question: Is this patient at high risk for diabetes?
    {patient_text}

    All experts will write down 1 step of their thinking, then share it with the group.
    Then all experts will go on to the next step. If any expert realizes they are wrong, they leave.
    Finally, the remaining experts agree on a conclusion on the very last line, formatted exactly as: "Final Conclusion: Positive" or "Final Conclusion: Negative".
    """
    response_tot = model.generate_content(tot_prompt).text.strip()
    prompt_results['ToT'].append(parse_llm_output(response_tot))
    time.sleep(0.5)

# Calculate and display the final accuracy for each prompt engineering method
print("\n" + "="*50)
print("PROMPT ENGINEERING ACCURACY RESULTS")
print("="*50)
for method in ['Zero-Shot', 'Few-Shot', 'CoT', 'ToT']:
    acc = accuracy_score(prompt_results['Actual'], prompt_results[method])
    print(f"{method:<12} Accuracy: {acc * 100:.2f}%")
print("="*50)


--- Running Prompt Engineering Evaluation on 11 Patients ---
This may take a minute due to API rate limits...



Evaluating Prompts: 100%|██████████| 11/11 [04:33<00:00, 24.84s/it]


PROMPT ENGINEERING ACCURACY RESULTS
Zero-Shot    Accuracy: 81.82%
Few-Shot     Accuracy: 90.91%
CoT          Accuracy: 90.91%
ToT          Accuracy: 90.91%


# Embeddings + Logistic Regression

In [54]:
print("\n--- Starting Bonus Section: Embeddings & Classification ---")

def get_gemini_embeddings(text_list):
    """Fetches embeddings from Gemini API for a list of text strings."""
    embeddings = []
    # Process in smaller batches to avoid rate limits
    for text in tqdm(text_list, desc="Generating Embeddings"):
        try:
            result = genai.embed_content(
                model="models/text-embedding-004", # Google's latest embedding model
                content=text,
                task_type="SEMANTIC_SIMILARITY"
            )
            embeddings.append(result['embedding'])
            time.sleep(0.5) # Sleep to respect free-tier API rate limits
        except Exception as e:
            print(f"Error fetching embedding: {e}")
            embeddings.append(np.zeros(768)) # Fallback zero-vector
    return np.array(embeddings)

# 1. Generate embeddings for Training and Testing sets
print("Fetching embeddings for the training set...")
X_train_embeddings = get_gemini_embeddings(train_df['Patient_Text'].tolist())
y_train = train_df['Diabetes_Risk'].values

print("Fetching embeddings for the test set...")
X_test_embeddings = get_gemini_embeddings(test_df['Patient_Text'].tolist())
y_test = test_df['Diabetes_Risk'].values

# 2. Train a standard Logistic Regression Classifier on the LLM Embeddings
print("\nTraining Logistic Regression model on Gemini Embeddings...")
classifier = LogisticRegression(max_iter=1000)
classifier.fit(X_train_embeddings, y_train)

# 3. Evaluate the model
y_pred = classifier.predict(X_test_embeddings)
y_prob = classifier.predict_proba(X_test_embeddings)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
auroc = roc_auc_score(y_test, y_prob)

print(f"\n--- Embedding Classification Results ---")
print(f"Accuracy: {accuracy * 100:.2f}%")
print(f"AUROC:    {auroc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Negative", "Positive"]))


--- Starting Bonus Section: Embeddings & Classification ---
Fetching embeddings for the training set...


Generating Embeddings:   5%|▍         | 2/43 [00:00<00:11,  3.62it/s]

Error fetching embedding: 404 POST https://generativelanguage.googleapis.com/v1beta/models/text-embedding-004:embedContent?%24alt=json%3Benum-encoding%3Dint: models/text-embedding-004 is not found for API version v1beta, or is not supported for embedContent. Call ListModels to see the list of available models and their supported methods.
Error fetching embedding: 404 POST https://generativelanguage.googleapis.com/v1beta/models/text-embedding-004:embedContent?%24alt=json%3Benum-encoding%3Dint: models/text-embedding-004 is not found for API version v1beta, or is not supported for embedContent. Call ListModels to see the list of available models and their supported methods.


Generating Embeddings:   9%|▉         | 4/43 [00:00<00:08,  4.45it/s]

Error fetching embedding: 404 POST https://generativelanguage.googleapis.com/v1beta/models/text-embedding-004:embedContent?%24alt=json%3Benum-encoding%3Dint: models/text-embedding-004 is not found for API version v1beta, or is not supported for embedContent. Call ListModels to see the list of available models and their supported methods.
Error fetching embedding: 404 POST https://generativelanguage.googleapis.com/v1beta/models/text-embedding-004:embedContent?%24alt=json%3Benum-encoding%3Dint: models/text-embedding-004 is not found for API version v1beta, or is not supported for embedContent. Call ListModels to see the list of available models and their supported methods.


...

Generating Embeddings:  91%|█████████ | 10/11 [00:01<00:00,  5.67it/s]

Error fetching embedding: 404 POST https://generativelanguage.googleapis.com/v1beta/models/text-embedding-004:embedContent?%24alt=json%3Benum-encoding%3Dint: models/text-embedding-004 is not found for API version v1beta, or is not supported for embedContent. Call ListModels to see the list of available models and their supported methods.


Generating Embeddings: 100%|██████████| 11/11 [00:01<00:00,  5.53it/s]

Error fetching embedding: 404 POST https://generativelanguage.googleapis.com/v1beta/models/text-embedding-004:embedContent?%24alt=json%3Benum-encoding%3Dint: models/text-embedding-004 is not found for API version v1beta, or is not supported for embedContent. Call ListModels to see the list of available models and their supported methods.

Training Logistic Regression model on Gemini Embeddings...

--- Embedding Classification Results ---
Accuracy: 72.73%
AUROC:    0.5000

Classification Report:
              precision    recall  f1-score   support

    Negative       0.00      0.00      0.00         3
    Positive       0.73      1.00      0.84         8

    accuracy                           0.73        11
   macro avg       0.36      0.50      0.42        11
weighted avg       0.53      0.73      0.61        11




/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
